# Human Face Annotation Summary

This notebook loads the manual page-level face annotations from `../../data/annotations/human_face_page_annotations.json`, validates the schema, and computes an inline summary for issues published after 1900. It also reports period totals from the processed decade-extremes face-count CSV: `../../data/processed/faces_per_issue_decade_extremes_through_2007.csv`.

The key rule is:

- `analyzable_faces`: usable for downstream smiling-behavior annotation;
- `non_analyzable_faces`: present but unusable because of image quality or face visibility.

No files are written by this notebook.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

annotations_json = Path("../../data/annotations/human_face_page_annotations.json")
assert annotations_json.exists(), f"Missing input file: {annotations_json}"

with annotations_json.open("r", encoding="utf-8") as fh:
    payload = json.load(fh)

assert payload["dataset_name"] == "human_face_page_annotations"
assert payload["schema_version"] == 1
assert isinstance(payload["issues"], list) and payload["issues"], "Issue list is empty."

issues = pd.DataFrame(payload["issues"])
expected_issue_columns = [
    "issue_id",
    "issue_date",
    "issue_year",
    "has_faces",
    "page_annotations",
    "issue_totals",
]
assert list(issues.columns) == expected_issue_columns, {
    "expected": expected_issue_columns,
    "actual": list(issues.columns),
}

issues["issue_date"] = pd.to_datetime(issues["issue_date"], format="%Y-%m-%d")
issues["issue_year"] = issues["issue_year"].astype("int64")

issue_totals = pd.json_normalize(issues["issue_totals"])
expected_total_columns = [
    "pages_with_faces",
    "analyzable_faces",
    "non_analyzable_faces",
    "total_faces_found",
]
assert list(issue_totals.columns) == expected_total_columns, {
    "expected": expected_total_columns,
    "actual": list(issue_totals.columns),
}

issue_summary = pd.concat([issues.drop(columns=["page_annotations", "issue_totals"]), issue_totals], axis=1)
issue_summary


In [ ]:
page_rows = []
for issue in payload["issues"]:
    for page in issue["page_annotations"]:
        page_rows.append(
            {
                "issue_id": issue["issue_id"],
                "issue_date": issue["issue_date"],
                "issue_year": issue["issue_year"],
                **page,
            }
        )

pages = pd.DataFrame(page_rows)
expected_page_columns = [
    "issue_id",
    "issue_date",
    "issue_year",
    "page_number",
    "analyzable_faces",
    "non_analyzable_faces",
    "total_faces_found",
]
assert list(pages.columns) == expected_page_columns, {
    "expected": expected_page_columns,
    "actual": list(pages.columns),
}

pages["issue_date"] = pd.to_datetime(pages["issue_date"], format="%Y-%m-%d")
pages["issue_year"] = pages["issue_year"].astype("int64")
numeric_columns = [
    "page_number",
    "analyzable_faces",
    "non_analyzable_faces",
    "total_faces_found",
]
pages[numeric_columns] = pages[numeric_columns].astype("int64")

assert (pages["total_faces_found"] == pages["analyzable_faces"] + pages["non_analyzable_faces"]).all()

issue_check = (
    pages.groupby("issue_id", as_index=False)
    .agg(
        pages_with_faces=("page_number", "size"),
        analyzable_faces=("analyzable_faces", "sum"),
        non_analyzable_faces=("non_analyzable_faces", "sum"),
        total_faces_found=("total_faces_found", "sum"),
    )
    .sort_values("issue_id")
    .reset_index(drop=True)
)

expected_issue_check = (
    issue_summary.loc[issue_summary["has_faces"], [
        "issue_id",
        "pages_with_faces",
        "analyzable_faces",
        "non_analyzable_faces",
        "total_faces_found",
    ]]
    .sort_values("issue_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(issue_check, expected_issue_check)
pages


In [ ]:
annotation_period_definitions = [
    ("after 1900", pages["issue_year"] > 1900),
    ("1940s", pages["issue_year"].between(1940, 1949, inclusive="both")),
]
assert all(mask.any() for _, mask in annotation_period_definitions), "One or more annotation period filters selected no rows."

annotation_period_summary = pd.DataFrame(
    [
        {
            "period": period,
            "issues_with_faces": int(pages.loc[mask, "issue_id"].nunique()),
            "pages_with_faces": int(mask.sum()),
            "faces_found": int(pages.loc[mask, "total_faces_found"].sum()),
            "analyzable_faces": int(pages.loc[mask, "analyzable_faces"].sum()),
            "non_analyzable_faces": int(pages.loc[mask, "non_analyzable_faces"].sum()),
        }
        for period, mask in annotation_period_definitions
    ]
)

print(annotation_period_summary.to_string(index=False))
annotation_period_summary


In [ ]:
extreme_issues_csv = Path("../../data/processed/faces_per_issue_decade_extremes_through_2007.csv")
assert extreme_issues_csv.exists(), f"Missing input file: {extreme_issues_csv}"

decade_extremes = pd.read_csv(extreme_issues_csv)
expected_extreme_columns = [
    "decade",
    "mode",
    "sample_order",
    "issue_id",
    "issue_date",
    "year",
    "detected_faces",
    "pages_with_faces",
]
assert list(decade_extremes.columns) == expected_extreme_columns, {
    "expected": expected_extreme_columns,
    "actual": list(decade_extremes.columns),
}

decade_extremes["year"] = decade_extremes["year"].astype("int64")
decade_extremes["detected_faces"] = decade_extremes["detected_faces"].astype("int64")

period_definitions = [
    ("before 1900", decade_extremes["year"] < 1900),
    ("1900-1939", decade_extremes["year"].between(1900, 1939, inclusive="both")),
    ("1940s", decade_extremes["year"].between(1940, 1949, inclusive="both")),
]
assert all(mask.any() for _, mask in period_definitions), "One or more period filters selected no rows."

face_period_sums = pd.DataFrame(
    [
        {
            "period": period,
            "sampled_issues": int(mask.sum()),
            "detected_faces": int(decade_extremes.loc[mask, "detected_faces"].sum()),
        }
        for period, mask in period_definitions
    ]
)

print(face_period_sums.to_string(index=False))
face_period_sums
